# Fine-tune RoBERTa for Fake News Detection
## Google Colab Training Notebook

This notebook fine-tunes RoBERTa on the WELFake dataset for fake news classification.

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers torch scikit-learn pandas tqdm

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted!")

## Step 3: Setup and Imports

In [ ]:
import os
import logging
import pandas as pd
import torch
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 4: Define Dataset Class

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("✓ NewsDataset class defined")

## Step 5: Define Evaluation Function

In [ ]:
def evaluate(model, loader, device):
    """Evaluate model on validation/test set."""
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().tolist())

    return {
        'accuracy':  round(accuracy_score(all_labels, all_preds), 4),
        'precision': round(precision_score(all_labels, all_preds, zero_division=0), 4),
        'recall':    round(recall_score(all_labels, all_preds, zero_division=0), 4),
        'f1':        round(f1_score(all_labels, all_preds, zero_division=0), 4),
    }, all_preds, all_labels

print("✓ Evaluate function defined")

## Step 6: Configure Training Parameters

In [ ]:
# ===== MODIFY THESE PARAMETERS =====

# Path to your CSV file in Google Drive
# Download from: https://www.kaggle.com/datasets/shoaib98/fake-news-dataset
DATA_PATH = '/content/drive/MyDrive/WELFake_Dataset.csv'

# Model and training config
MODEL_NAME = 'roberta-base'
EPOCHS = 3
BATCH_SIZE = 16  # Increase to 32 for more GPU memory
LEARNING_RATE = 2e-5
MAX_LEN = 256

# Output directory in Google Drive
OUTPUT_DIR = '/content/drive/MyDrive/roberta_fine_tuned'

print(f"Data path: {DATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Model: {MODEL_NAME}")
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}")

## Step 7: Load and Preprocess Data

In [ ]:
logger.info("Loading data...")
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['text', 'label'])

logger.info(f"Total samples: {len(df)}")

# Map labels
label_map = {'fake': 0, 'FAKE': 0, '0': 0, 'real': 1, 'REAL': 1, '1': 1}
df['label'] = df['label'].astype(str).map(label_map)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

logger.info(f"Samples after cleaning: {len(df)}")
logger.info(f"\nLabel distribution:")
print(df['label'].value_counts())

## Step 8: Split Data

In [ ]:
# Split: 70% train, 15% val, 15% test
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
)

logger.info(f"Train: {len(train_df)}")
logger.info(f"Val:   {len(val_df)}")
logger.info(f"Test:  {len(test_df)}")

## Step 9: Load Model and Tokenizer

In [ ]:
logger.info(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)
logger.info("✓ Model loaded")

## Step 10: Create DataLoaders

In [ ]:
logger.info("Creating datasets...")
train_dataset = NewsDataset(train_df['text'].values, train_df['label'].values, tokenizer, MAX_LEN)
val_dataset = NewsDataset(val_df['text'].values, val_df['label'].values, tokenizer, MAX_LEN)
test_dataset = NewsDataset(test_df['text'].values, test_df['label'].values, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

logger.info(f"✓ DataLoaders created")

## Step 11: Setup Optimizer and Scheduler

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1.0, end_factor=0.0, total_iters=total_steps
)

logger.info(f"✓ Optimizer and scheduler configured")

## Step 12: Train Model

In [ ]:
logger.info("\n" + "="*80)
logger.info("TRAINING STARTED")
logger.info("="*80)

best_f1 = 0

for epoch in range(EPOCHS):
    logger.info(f"\n{'='*80}")
    logger.info(f"EPOCH {epoch+1}/{EPOCHS}")
    logger.info(f"{'='*80}")
    
    # Training phase
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader, desc=f"Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    avg_loss = total_loss / len(train_loader)
    
    # Validation phase
    logger.info(f"\nValidating...")
    val_metrics, _, _ = evaluate(model, val_loader, device)
    
    logger.info(f"Loss:      {avg_loss:.4f}")
    logger.info(f"Accuracy:  {val_metrics['accuracy']:.4f}")
    logger.info(f"Precision: {val_metrics['precision']:.4f}")
    logger.info(f"Recall:    {val_metrics['recall']:.4f}")
    logger.info(f"F1 Score:  {val_metrics['f1']:.4f}")

    # Save best model
    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        logger.info(f"✓ Best model saved (F1={best_f1:.4f})")

logger.info(f"\n{'='*80}")
logger.info("TRAINING COMPLETED")
logger.info(f"{'='*80}")

## Step 13: Evaluate on Test Set

In [ ]:
logger.info(f"\n{'='*80}")
logger.info("TEST SET EVALUATION")
logger.info(f"{'='*80}")

test_metrics, test_preds, test_labels = evaluate(model, test_loader, device)

logger.info(f"\nTest Metrics:")
logger.info(f"Accuracy:  {test_metrics['accuracy']:.4f}")
logger.info(f"Precision: {test_metrics['precision']:.4f}")
logger.info(f"Recall:    {test_metrics['recall']:.4f}")
logger.info(f"F1 Score:  {test_metrics['f1']:.4f}")

## Step 14: Confusion Matrix and Classification Report

In [ ]:
logger.info(f"\nConfusion Matrix:")
cm = confusion_matrix(test_labels, test_preds)
logger.info(f"\n{cm}")

logger.info(f"\nClassification Report:")
report = classification_report(test_labels, test_preds, target_names=['Fake', 'Real'])
print(report)

## Step 15: Summary

In [ ]:
logger.info(f"\n{'='*80}")
logger.info("✓ TRAINING COMPLETE!")
logger.info(f"{'='*80}")
logger.info(f"Model saved to: {OUTPUT_DIR}")
logger.info(f"Best F1 Score on Test Set: {test_metrics['f1']:.4f}")
logger.info(f"Best F1 Score on Validation: {best_f1:.4f}")